In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas_datareader import data as pdr
from datetime import datetime, timedelta
import statsmodels.api as sm
from scipy.optimize import minimize
import os
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [ ]:
class Utils:
  @staticmethod
  def read_data(ticker, start, end, constrains):
    try:
      t = yf.Ticker(ticker)
      df = t.history(start=start, end=end, interval="1mo")[['Close', 'Volume']]
      df.index = df.index.tz_localize(None).normalize()

      if not Utils.check_constrains(df["Close"].mean(), df["Volume"].mean(), constrains):
        return False

      annual_bs = t.balance_sheet

      target_rows = ['Stockholders Equity', 'Share Issued']
      financials = annual_bs.loc[annual_bs.index.intersection(target_rows)].T

      financials.index = pd.to_datetime(financials.index).tz_localize(None).normalize()

      df = pd.merge_asof(
        df.sort_index(),
        financials.sort_index(),
        left_index=True,
        right_index=True,
        direction='backward'
      )

      df["Market_Cap"] = df["Share Issued"] * df["Close"]
      
      df["BMV"] = df["Stockholders Equity"] / df["Market_Cap"]

      df["Momentum"] = df["Close"].pct_change(periods=12)

      df["Returns"] = df["Close"].pct_change()

    except Exception as e:
      raise Exception(f"Error retrieving data for {ticker}: {e}")

    else:
      return (
          df["Returns"].values,
          df["Market_Cap"].values,
          df["BMV"].values,
          df["Momentum"].values
      )

  def check_constrains(close_mean, volume_mean, constrains):
    volume_mean = volume_mean / 1000000

    if constrains is None:
      return True

    if close_mean < constrains["min_price"] or volume_mean < constrains["min_volume"]:
      return False

    return True

  @staticmethod
  def compute_z_scores(x):
    mu = np.mean(x)
    sigma = np.std(x)

    z = (x - mu) / sigma

    return z, sigma

  @staticmethod
  def dot_product(A, B):
    return np.sum(A * B, axis=1).reshape(-1, 1)



In [ ]:
class DataReader:
  def __init__(self, tickers, start, end, constrains, reader_params):
    self.start = start
    self.end = end
    self.tickers = tickers
    self.reader_params = reader_params
    self.factors = None
    self.constrains = constrains
    self.market_data = None


  def generate_data(self):
    self.factors = self.read_factors(reader_params)
    data = self.read_market_data(
        self.tickers, self.start,
        self.end, self.constrains
    )
    self.market_data, self.BM, self.size, self.momentum = data

  def read_factors(self, reader_params):
    factors = pdr.DataReader(*self.reader_params)
    factor_data = factors[0]
    factor_data = factor_data[self.start:self.end]
    return factor_data

  def read_market_data(self, tickers, start, end, constrains):
    price_data = {}
    size = {}
    momentum = {}
    BMs = {}

    for ticker in tickers:
      res = Utils.read_data(ticker, start, end, constrains)

      if not res:
        continue

      returns, market_cap, bm, mm = res

      price_data[ticker] = returns
      size[ticker] = np.log(market_cap)
      BMs[ticker] = bm
      momentum[ticker] = mm

    sizes = pd.DataFrame(size).dropna()
    BM = pd.DataFrame(BMs).dropna()
    mm = pd.DataFrame(momentum).dropna()

    
    for k in price_data.keys():
      print(f"{k}: {len(price_data[k])}")

    m_data = pd.DataFrame(price_data).dropna()

    return m_data, BM, sizes, mm

  @property
  def datastore(self):
    return {
      "BM": self.BM,
      "factors": self.factors,
      "market_data": self.market_data,
      "size": self.size,
      "momentum": self.momentum
    } if self.market_data is not None else None



In [ ]:
class FeatureStore(DataReader):
  def __init__(self, tickers, start, end, cons, reader_params, debug:bool=False):
    super().__init__(tickers, start, end, cons, reader_params)
    self.debug = debug

  def load_or_generate_data(self):
    returns_path = f"returns_{self.start}_{self.end}.parquet"
    factors_path = f"factors_{self.start}_{self.end}.parquet"
    scores_path = f"scores{self.start}_{self.end}.parquet"

    if os.path.exists(returns_path) and os.path.exists(factors_path) and os.path.exists(scores_path) and not self.debug:
      if self.debug:
        print("loading data")

      self.scores = pd.read_parquet(scores_path)
      self.returns = pd.read_parquet(returns_path)
      self.factors = pd.read_parquet(factors_path)

      if self.debug:
        print(self.scores.head())
        print(self.returns.head())
        print(self.factors.head())

    else:
      if self.debug:
        print("generating data")

      self.generate_data()

      self.returns, self.factors = self.construct_factors()
      self.scores = self.compute_scores()
      self.scores.to_parquet(scores_path)
      self.returns.to_parquet(returns_path)
      self.factors.to_parquet(factors_path)

    return self

  def compute_scores(self):
    omegas = {}
    raw_signals = {}

    for i in ['size', "BM", "momentum"]:
      print(f"\n\n {i} \n\n")
      x = self.datastore[i]
      coef = -1 if i == 'size' else 1
      raw_signals[i], sigma = Utils.compute_z_scores(x)
      raw_signals[i] *= coef
      omegas[i] = 1/sigma

    print(raw_signals)
    raw_signals_df = pd.DataFrame(raw_signals).T
    omegas = pd.Series(omegas)


    scores = pd.DataFrame(raw_signals_df.mul(omegas, axis=0).sum())

    return scores


  def construct_factors(self):
    returns = self.datastore["market_data"].copy()
    factors = self.datastore["factors"].copy()

    rf = factors["RF"]
    factors = factors.drop(columns=["RF"])

    common_dates = returns.index.intersection(rf.index)
    returns = returns.loc[common_dates]
    rf = rf.loc[common_dates]


    returns = returns.sub(rf, axis=0)

    return returns, factors


In [ ]:
class FactorModel:
  def __init__(self, alpha=10.0, err_lambda=0.94, window_size=12, solver="auto", min_var=None):
    self.alpha = alpha
    self.window_size = window_size
    self.solver = solver
    self.lam = err_lambda
    self.epsilon_min = min_var

  def calculate_window_beta(self, x, y):
    model = Ridge(alpha=self.alpha, fit_intercept=True, solver=self.solver)

    model.fit(x, y)
    beta = model.coef_
    intercept = model.intercept_
    res = (y - model.predict(x))**2
    sigma2 = res[0]

    for t in range(1, len(res)):
      sigma2 = self.lam * res[t] + (1 - self.lam) * sigma2

    if self.epsilon_min:
      sigma2 = max(sigma2, self.epsilon_min)

    return (intercept, beta, sigma2)

  def fit(self, X, returns):
    y = returns.copy()

    factors_list = X.columns
    tickers = y.columns

    n_rows = len(y)
    n_cols = len(tickers) * len(factors_list)

    np_betas = np.empty((n_rows, n_cols))
    np_betas[:] = np.nan
    np_alphas = np.empty((n_rows, len(tickers)))
    np_alphas[:] = np.nan
    np_idiosyncratic_var = np.empty((n_rows, len(tickers)))
    np_idiosyncratic_var[:] = np.nan

    cols = pd.MultiIndex.from_product([tickers, factors_list], names=['Ticker', 'Factor'])

    for i, ticker in enumerate(tickers):
      y_series = y[ticker]
      for j in range(self.window_size - 1, n_rows):
          x_window = X.iloc[j-self.window_size+1 : j+1]
          y_window = y_series.iloc[j-self.window_size+1 : j+1]

          results = self.calculate_window_beta(x_window, y_window)

          (alpha, beta, sigma2) = results[0], results[1], results[2]

          start_col = i * len(factors_list)
          end_col = start_col + len(factors_list)

          np_betas[j, start_col:end_col] = beta
          np_alphas[start_col:end_col] = alpha
          np_idiosyncratic_var[start_col:end_col] = sigma2

    betas = pd.DataFrame(np_betas, columns=cols, index=y.index).dropna()
    alphas = pd.DataFrame(np_alphas, columns=y.columns, index=y.index).dropna()
    idiosyncratic_vars = pd.DataFrame(np_idiosyncratic_var, columns=y.columns, index=y.index).dropna()

    return alphas, betas, idiosyncratic_vars




In [ ]:
class FactorPremiaModel:
  def __init__(self, alpha=0.5, lam_alpha=0.4):
   self.factors = ["Mkt-RF",	"SMB",	"HML",	"RMW",	"CM"]
   self.alpha = alpha
   self.lam_alpha = lam_alpha

  def normalize(self, df):
    return df.rolling(1).apply(lambda x: Utils.compute_z_scores(x)).dropna()

  def estimate_lambda(self, factors, alpha):
    factor_means = np.mean(factors, axis=1)
    return alpha * factor_means

  def compute_expected_return(self, betas, scores, factor_returns):
    n_rows = betas.shape[0]
    n_cols = betas.shape[1]
    cols = betas.columns

    scores = Utils.compute_z_scores(scores)

    ER_i_empty = np.empty((n_rows, n_cols))
    ER_i_empty[:] = np.nan
    ER_i = pd.DataFrame(ER_i_empty, columns=cols, index=betas.index)

    beta_scaled = {}

    for ticker in cols:
      beta_i = betas[ticker]

      lam = self.estimate_lambda(factor_returns, self.lam_alpha)

      beta_scaled[ticker] = Utils.dot_product(beta_i, lam)

    betas_df = self.normalize(pd.DataFrame(beta_scaled))
    scores_df = self.normalize(scores)


    for ticker in cols:
      score_i = scores[ticker]
      beta_i = betas_df[ticker]

      ER_i[ticker] = self.alpha * score_i + (1 - self.alpha) * beta_scaled[ticker]

    return ER_i





In [ ]:
from dataclasses import dataclass

@dataclass
class Constrains:
  investment: float = 1
  max_weight: float = 0.05
  turnover_lim: float = 0.2
  min_weight: float = 0
  beta_dev: float = .1

In [ ]:
class PortfolioObjective:
  def __init__(
      self, lam, betas, factor_premia,
      constrains, factor_penalty, cov,
      turnover_penalty, desired_exp,
      w_prew, use_factor_penalty=True
    ):

    self.lam = lam
    self.betas = betas
    self.factor_premia = factor_premia
    self.cov = cov
    self.constrains = constrains
    self.penalty = factor_penalty
    self.desired_exp = desired_exp
    self.turnover_penalty = turnover_penalty
    self.w_prev = w_prew

  def magnitude(self, x):
    x2_sum = np.sum(x**2)
    return np.sqrt(x2_sum)

  def loss(self, x):
    factor_target = self.penalty * (self.magnitude(self.beta.T - self.desired_exp))**2 if use_factor_penalty else 0
    turnover_penalty = self.turnover_penalty * (self.magnitude(x - self.w_prev))**2
    return x.T * self.factor_premia - self.lam * x  * self.cov * x.T - factor_target - turnover_penalty



In [ ]:
from scipy.optimize import minimize, NonlinearConstraint, LinearConstraint

class Optimizer:
  def __init__(self, obj, lam, betas, factor_premia, cov,
      constrains, factor_penalty,
      turnover_penalty, desired_exp, w_prew):
    self.obj = obj(lam, betas, factor_premia, cov, constrains, factor_penalty, turnover_penalty, desired_exp, w_prew)
    self.factor_opt = None
    self.cons = cons
    self.betas = betas
    self.w_prew = w_prew

  def get_constrains(self, x):
    n = len(x)

    A_sum = np.ones((1, n))
    lb_sum = [1]
    ub_sum = [1]
    w_sum_cons = LinearConstraint(A_sum, lb_sum, ub_sum)


    each_element = lambda x: x
    ub_size = [self.cons.max_weight or 1] * n
    lb_size = [self.cons.min_weight or 0] * n
    min_max_w_cons = NonlinearConstraint(each_element, lb_size, ub_size)

    lb_beta = [1-self.cons.beta_dev]
    ub_beta = [1+self.cons.beta_dev]
    beta_cons = w_sum_cons = LinearConstraint(self.betas, lb_beta, ub_beta)


    A_up = np.hstack([np.eye(n), -np.eye(n)])
    ub_up = w_prev

    A_low = np.hstack([-np.eye(n), -np.eye(n)])
    ub_low = -w_prew

    A_turn = np.hstack([np.zeros(n), np.ones(n)])
    ub_turn = [self.cons.turnover_lim]

    A_final = np.vstack([A_up, A_turn, A_low])
    ub_turnover = np.concatenate(ub_up, ub_turn, up_low)
    lb_turnover = np.full_like(ub_turnover, -np.inf)

    turnover_cons = LinearConstraint(A_final, ub_turnover, lb_turnover)

    return [w_sum_cons, min_max_w_cons, beta_cons, turnover_cons]


  def optimize(self, init = None,
               method: str="SLSQP") -> np.ndarray:
    n = len(self.obj)

    constraints = get_constrains

    results = minimize(
        fun=self.obj.loss,
        x0=init,
        method=method,
        constraints=constraints
    )



    if not results.success:
      raise RuntimeError("Optimization Failed")

    self.factor_opt = results.x
    return self.factor_opt

In [ ]:
class HighRiskPortfolio():
  def __init__(self, tickers:list, start:str, end:str, constrains:PortfolioObjective,
               reader_params:tuple[str], ridge_alpha:float=10.0, premia_alpha:float=0.5,
               lam_alpha:float=0.3, window:int=12, save_model: bool = False,
               model_path: str|None=None, debug:bool=False, cov_window=6):

    self.dataStore = FeatureStore(
        tickers = tickers,
        start=start,
        end=end,
        cons=constrains,
        reader_params=reader_params,
        debug=debug
    )

    self.cov_window = cov_window

    self.model = FactorModel(alpha=ridge_alpha, window=window)
    self.premia_model = FactorModel(alpha=premia_alpha, lam_alpha=lam_alpha)

  def calculate_covariance(self, ids_risk):
    idx = self.dataStore.returns.index
    n_rows = len(idx)
    n_cols = 1
    cov_empty = np.empty((n_rows, n_cols))
    cov_empty[:] = np.nan
    cov = pd.Series(cov_empty, index=idx)

    factor_cov = self.dataStore.factors.rolling(12).cov()

    for i in range(self.cov_window-1, len(self.dataStore.returns.index)):
      betas_i = betas.iloc[i].unstack(level='Factor').T
      D_i = ids_risk.iloc[i]
      factor_cov = self.dataStore.factors.iloc[i-self.cov_window+1:i+1].cov()

      cov.iloc[i] = betas_i * factor_cov * betas_i.T + D_i

    return cov

  def generate_optim_features(self):
    self.dataStore.load_or_generate_data()
    alphas, betas, ids_risk = self.model.fit(self.dataStore.factors, self.dataStore.returns)
    factor_returns = self.dataStore.factors.pct_change().dropna()
    factor_premia = self.premia_model.fit(betas, self.dataStore.signals, factor_returns)
    cov = self.calculate_covariance(ids_risk)

    return betas, factor_premia, cov

  def optimize_portfolio(self, betas, factor_premia, cov, constrains):
    pass

  def build_portfolios(self, constrains, ):
    betas, factor_premia, cov = self.generate_optim_features()
    self.optimize_portfolio(betas, factor_premia, cov, constrains)












In [ ]:
class PortfolioConstruction:
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    pass